In [1]:
import pandas as pd


In [2]:
import numpy as np


In [3]:
df=pd.read_csv("bengaluru_house_prices.csv")
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [5]:
import matplotlib.pyplot as plt

In [6]:
def convert_sqft(x):
    try:
        if isinstance(x, str):
            x = x.strip()
            
            # Case 1: range
            if '-' in x:
                a, b = x.split('-')
                return (float(a) + float(b)) / 2
            
            # Case 2: remove units like Sq. Yards, Sq. Meter
            if 'Sq.' in x:
                return None  # ignore these for now
            
            # Case 3: normal number
            return float(x)
        
        return x
    except:
        return None
    

In [7]:
df = df.copy()
df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df = df.dropna()

In [8]:
def extract_bhk(x):
    try:
        return int(str(x).split(' ')[0])
    except:
        return None

df['bhk'] = df['size'].apply(extract_bhk)


In [9]:
X=df[['total_sqft','bath','bhk','location']]
Y=df["price"]

In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [26]:
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.linear_model import LinearRegression

In [27]:
preprocessor = ColumnTransformer([
    ("num",StandardScaler(),["total_sqft","bath","bhk"]),
    ("cat",OneHotEncoder(handle_unknown="ignore"),['location'])
])

Pipe=Pipeline([
    ('preprocessing',preprocessor),
    ('model',LinearRegression())
])

Pipe.fit(X_train,Y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [28]:
print(X_train.columns)
df.columns

Index(['total_sqft', 'bath', 'bhk', 'location'], dtype='object')


Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price', 'bhk'],
      dtype='object')

In [29]:
print(X_train.head())

       total_sqft  bath  bhk             location
732        1688.0   3.0    3             Panathur
2871       1274.0   2.0    2        BTM 2nd Stage
1381        850.0   1.0    2           Chandapura
709        1045.0   2.0    2  Green Garden Layout
11226      3453.0   4.0    4           Whitefield


In [33]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(Pipe, f)

In [34]:
print(type(Pipe))

<class 'sklearn.pipeline.Pipeline'>


In [36]:
import pandas as pd

sample = pd.DataFrame({
    'total_sqft': [1000],
    'bath': [2],
    'bhk': [2],
    'location': ['Whitefield']
})

Pipe.predict(sample)

array([66.87048253])

In [38]:
prediction = Pipe.predict(sample)
print(f"Estimated Price: ₹ {prediction[0]:.2f} Lakhs")

Estimated Price: ₹ 66.87 Lakhs
